# Classification

## Configuration

In [ ]:
import sys
sys.path.append('../src')

In [ ]:
CONFIGURATION = '../configuration'

import configparser
import os
conf = configparser.ConfigParser()
conf.read((
    os.path.join(CONFIGURATION, f"default.conf"),
    os.path.join(CONFIGURATION, f"{os.uname().nodename}.conf")
))
conf.sections()

## Loading training data

In [ ]:
DIR_DATA = '../.data/split'
CONDITIONS = ('neutral','stress')
DURATIONS = (0,1,2,4,6,8)
CLASS_NAMES = ('understanding', 'confusion')

from trainer import DataProvider
data = DataProvider(DIR_DATA, CONDITIONS, DURATIONS, CLASS_NAMES)

In [ ]:
from sklearn import metrics
METRICS = (
    (
        'Accuracy', 
        lambda a,b: metrics.accuracy_score(a,b)
    ),
    (
        'Precision',
        lambda a,b: metrics.precision_score(a, b, average='macro', zero_division=0)
    ),
    (
        'Recall',
        lambda a,b: metrics.recall_score(a, b, average='macro', zero_division=0)
    ),
    (
        'ROC-AUC',
        lambda a,b: metrics.roc_auc_score(a, b, average='macro')
    ),
    (
        'F1-Score',
        lambda a,b: metrics.f1_score(a, b, average='macro', zero_division=0)
    ),
)

from trainer import Trainer
trainer = Trainer(data, METRICS)
print(len(trainer.labels_condition), trainer.labels_condition)
print(len(trainer.labels_duration), trainer.labels_duration)
print(len(trainer.labels_metric), trainer.labels_metric)
print(len(trainer.labels_eval), trainer.labels_eval)


## Models

In [ ]:

from models import FunctionalWrapper, MultiLabelLSTM, MultiLabelMLP
from sklearn.dummy import DummyClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Configuration
## Random forests
N_ESTIMATORS_RF = int(conf['Models']['rf-estimators'])
N_ESTIMATORS_XGB = int(conf['Models']['xgb-estimators'])
## Neural network
N_LAYERS = int(conf['Models']['mlp-hidden-layers'])
N_NEURONS = int(conf['Models']['mlp-layer-neurons'])

# Model dict
models = {
    '-Prior': DummyClassifier(strategy='prior'),
    '-Strat': DummyClassifier(strategy='stratified'),
    '-UForm': DummyClassifier(strategy='uniform'),
    'NBGauss': FunctionalWrapper(MultiOutputClassifier(GaussianNB())),
    'RandFor': FunctionalWrapper(RandomForestClassifier(N_ESTIMATORS_RF, n_jobs=16)),
    'XGBoost': FunctionalWrapper(XGBClassifier(eval_metric='logloss', n_estimators=N_ESTIMATORS_XGB, device='cuda')),
    'MLP': MultiLabelMLP(hidden_layers=((N_NEURONS,) * N_LAYERS)),
    'LSTM': MultiLabelLSTM(hidden_size=N_NEURONS, num_layers=N_LAYERS)
}

## Experiments

**In-Domain**

In [ ]:
model_scores = dict()
for name, model in models.items():
    print(name)
    trainer.model = model
    model_scores[name] = trainer.eval_in_and_cross_domain()

In [ ]:
import matplotlib.pyplot as plt
def plot_scores(score_dict, title, d_train=0, d_eval=0):
    fig, axs = plt.subplots(
        len(trainer.labels_eval), len(trainer.labels_metric), figsize=(12,8),
        sharex=True, sharey=True
    )
    for i, (eval, ax) in enumerate(zip(trainer.labels_eval, axs)):
        for j, (metric, a) in enumerate(zip(trainer.labels_metric, ax)):
            for name, scores in score_dict.items():
                a.plot(
                    trainer.durations,
                    scores[:,d_train,d_eval,j,i],
                    marker='o', label=name
                )
                a.set_xticks(trainer.durations)
                if i == 0:
                    a.set_title(metric)
                if i == len(trainer.labels_eval)-1:
                    a.set_xlabel('Segment duration (s)')
                if j == 0:
                    a.set_ylabel(eval)

    fig.legend(labels=list(score_dict.keys()), loc='right', fontsize='small')
    plt.suptitle(title)
    plt.ylim(0,1)
    plt.show()

In [ ]:
plot_scores(
    model_scores,
    'Model performance in neutral domain',
    d_train=0,
    d_eval=0
)

In [ ]:
plot_scores(
    model_scores,
    'Model performance in stress domain',
    d_train=1,
    d_eval=1
)

**Leave one out Corss-Validation**

In [ ]:
# Load previous scores
model_scores = dict()
import pickle
with open('./scores_new.pkl', 'rb') as file:
    model_scores = pickle.load(file)

In [ ]:
# Cross validation per model
for name, model in models.items():
    print(name)
    trainer.model = model
    model_scores[name] = trainer.eval_in_domain_cross_validation('neutral')

In [ ]:
# save cross validation results
import pickle
with open('./scores.pkl', 'wb+') as file:
    pickle.dump(model_scores, file)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
scores = np.array([
    [
        [
            s
            for duration, s in series.items()
        ]
        for token, series in scores.items()
    ]
    for model, scores in model_scores.items()
])
mean_scores = scores.mean(axis=1)

fig, axs = plt.subplots(
    len(trainer.labels_eval), len(trainer.labels_metric), figsize=(12,8),
    sharex=True, sharey=True
)
for i, (eval, ax) in enumerate(zip(trainer.labels_eval, axs)):
    for j, (metric, a) in enumerate(zip(trainer.labels_metric, ax)):
        for name, scores in zip(model_scores.keys(), mean_scores):
            a.plot(
                trainer.durations,
                scores[:,j,i],
                marker='o', label=name
            )
            a.set_xticks(trainer.durations)
            if i == 0:
                a.set_title(metric)
            if i == len(trainer.labels_eval)-1:
                a.set_xlabel('Segment duration (s)')
            if j == 0:
                a.set_ylabel(eval)

fig.legend(labels=list(model_scores.keys()), loc='right', fontsize='small')
plt.suptitle('In domain neutral cross validation')
plt.ylim(0,1)
plt.show()

**Data Restructure**

In [ ]:
# combine train + test
import numpy as np
import matplotlib.pyplot as plt
def combine(condition, duration, sets=['train', 'test']):
    source = data._get(condition, duration)
    x, y, t = map(np.concat, zip(*tuple(source[s] for s in sets)))
    return x, y,t
    
x, y, t = combine('neutral', 0)
import pandas as pd
class_distribution = pd.DataFrame(
    y[:,0],
    columns=['understanding', 'confusion']
).groupby(t).mean().plot(
    kind='box',
    label='understanding',
    title='Distribution of participnat-wise relative samples per class'
)
plt.ylim(0,1)

In [ ]:
from trainer import DataProviderInterParticipant
data_part = DataProviderInterParticipant(DIR_DATA, CONDITIONS, DURATIONS, CLASS_NAMES)

In [ ]:
trainer = Trainer(data_part, METRICS)

In [ ]:
model_scores = dict()
for name, model in models.items():
    print(name)
    trainer.model = model
    model_scores[name] = trainer.eval_in_and_cross_domain()

In [ ]:
plot_scores(
    model_scores,
    'Model performance in neutral domain',
    d_train=0,
    d_eval=0
)

In [ ]:
plot_scores(
    model_scores,
    'Model performance in stress domain',
    d_train=1,
    d_eval=1
)

**Functional MLP**

In [ ]:
functional_scores = model_scores.copy()
del functional_scores['-Prior']
del functional_scores['-Strat']
del functional_scores['-UForm']
del functional_scores['LSTM']
trainer.model = FunctionalWrapper(models['MLP'])
functional_scores['MLPFunc'] = trainer.eval_in_and_cross_domain()

In [ ]:
plot_scores(
    functional_scores,
    'Model with Functionals in neutral domain',
    d_train=0,
    d_eval=0
)

In [ ]:
plot_scores(
    functional_scores,
    'Model with Functionals in stress domain',
    d_train=1,
    d_eval=1
)

**Cross-doamin**

In [ ]:
plot_scores(
    model_scores,
    'Model performance cross domain (train neutral, eval stress)',
    d_train=0,
    d_eval=1
)

In [ ]:
plot_scores(
    model_scores,
    'Model performance cross domain (train stress, eval neutral)',
    d_train=1,
    d_eval=0
)

**All-domain**


In [ ]:
full_scores = dict()
for name, model in models.items():
    print(name)
    trainer.model = model
    full_scores[name] = trainer.eval_all_domains()

In [ ]:
import matplotlib.pyplot as plt
fig, axs = plt.subplots(
    len(trainer.labels_eval), len(trainer.labels_metric), figsize=(12,8),
    sharex=True, sharey=True
)
for i, (eval, ax) in enumerate(zip(('neutral', 'stress', 'total'), axs)):
    for j, (metric, a) in enumerate(zip(trainer.labels_metric, ax)):
        for name, scores in full_scores.items():
            a.plot(
                trainer.durations,
                scores[:,i,j,0],
                marker='o', label=name
            )
            a.set_xticks(trainer.durations)
            if i == 0:
                a.set_title(metric)
            if i == len(trainer.labels_eval)-1:
                a.set_xlabel('Segment duration (s)')
            if j == 0:
                a.set_ylabel(eval)

fig.legend(labels=list(full_scores.keys()), loc='right', fontsize='small')
plt.suptitle('Trained on data from all domains')
plt.ylim(0,1)
plt.show()

**Single-frame vs Context**

In [ ]:
x_train,y_train = data_part.get_train('neutral', 1, agg_window=False)
x_eval,y_eval = data_part.get_eval('neutral', 1, agg_window=False)
print(x_train.shape, y_train.shape)
print(x_eval.shape, y_eval.shape)

Full Window MLP

In [ ]:
from models import MultiLabelMLP
model = MultiLabelMLP((600,) * 4)
model.fit(
    x_train,
    (y_train.mean(axis=1) > .5).astype(float)
)
y_pred = model.predict(x_eval)
for name, func in METRICS:
    print(name, trainer._score(
        (y_eval.mean(axis=1) > .5).astype(float),
        y_pred,
        func))

Single Frame MLP

In [ ]:
model.fit(
    x_train.reshape(-1, 1, 18),
    y_train.reshape(-1, 2)
)
y_pred = model.predict(x_eval.reshape(-1, 1, 18)).reshape(*y_eval.shape)
for name, func in METRICS:
    print(name, trainer._score(
        (y_eval.mean(axis=1) > .5).astype(float),
        (y_pred.mean(axis=1) > .5).astype(float),
        func))

Functional pooled MLP

In [ ]:
from models import FunctionalWrapper
model = FunctionalWrapper(MultiLabelMLP((600,) * 4))
model.fit(
    x_train,
    (y_train.mean(axis=1) > .5).astype(float)
)
y_pred = model.predict(x_eval)
for name, func in METRICS:
    print(name, trainer._score(
        (y_eval.mean(axis=1) > .5).astype(float),
        y_pred,
        func))

LSTM

In [ ]:
from models import MultiLabelLSTM
model = MultiLabelLSTM(
    hidden_size=600,
    num_layers=4
)
model.fit(
    x_train,
    (y_train.mean(axis=1) > .5).astype(float)
)
y_pred = model.predict(x_eval)
for name, func in METRICS:
    print(name, trainer._score(
        (y_eval.mean(axis=1) > .5).astype(float),
        y_pred,
        func))

In [ ]:
from models import Seq2SeqLSTM
model = Seq2SeqLSTM(
    hidden_size=600,
    num_layers=4
)
model.fit(
    x_train,
    y_train
)
y_pred = model.predict(x_eval)
for name, func in METRICS:
    print(name, trainer._score(
        (y_eval.mean(axis=1) > .5).astype(float),
        (y_pred.mean(axis=1) > .5).astype(float),
        func))